# Indiana Bosonic Code - DMANH

In [27]:
from jaqalpaq import run
from jaqalpaq import emulator
from jaqalpaq.run import run_jaqal_file, run_jaqal_string, run_jaqal_batch, run_jaqal_circuit, frontend

from jaqalpaq.emulator.unitary import UnitarySerializedEmulator
emulator_backend = UnitarySerializedEmulator()

#from Experiment import Experiment
import numpy as np
import csv
from datetime import datetime, date, time, timezone
from pytz import timezone
from collections import OrderedDict
import pytz
from scipy.special import jn_zeros
import math
import matplotlib.pyplot as plt
import pickle as pkl
from jaqalpaq.parser import parse_jaqal_string


def timestamp_generate():
    mountain = timezone('US/Mountain')
    timestamp_utc = datetime.utcnow()
    timestamp_local = timestamp_utc.astimezone(mountain)
    return(timestamp_utc, timestamp_local)

def int_to_base(n, b):
    if n == 0:
        return "0"
    digits = []
    is_negative = n < 0
    n = abs(n)
    while n:
        digits.append(int(n % b))
        n //= b
    if is_negative:
        digits.append("-")
    return ''.join(str(x) if x < 10 else chr(55 + x) for x in reversed(digits))



In [ ]:
#Can import a file here with the angles in it, so angles can be called automatically. 

repeats = 1000
override_dict={ "__repeats__":repeats,}
#beta_list = [0]

import os
import ast

working_dir = os.getcwd()
recursive_dir = working_dir
for ii in range(5):
    temp_dir = os.path.dirname(recursive_dir)
    recursive_dir = temp_dir
file_path = recursive_dir


#with open(os.path.join(file_path,'Indiana_QSD_Dict1.txt'), 'r') as file:
#    string_data = file.read()
#angles = ast.literal_eval(string_data)

# ok, making this part up now as it doesn't exist, so how 
# exactly this is called in should be modified to match the actual file
#angle = [[-0.18512000000000001,0, -0.79999999999999982],[-0.18128949140032261,-0.03746377861097782, -0.79999999999999982],[-0.16995648759926149, -0.073377153957632377,-0.79999999999999982]]
angles = np.load('../build/notebook/angles.npy')  # shape (3, 49)
# betas should be: 0 -0.4 0 -0.2 0 0 0 0.2 0 0.4
betas = np.load('../build/notebook/betas.npy')  # shape (N, 2)
beta_list = [complex(b[0], b[1]) for b in betas]

#timestep = 3 #length of the code
timesteps = angles.shape[1]


def generate_jaqal_header(beta):
    sandia_beta = 0.5j * beta
    header = '''from Calibration_PulseDefinitions.QubitBosonPulses usepulses *

let reBeta {sandia_beta.real}
let imBeta {sandia_beta.imag}
let imMeas 1

register q[2]

// I added this, assuming it needs to be there? - JBG
prepare_all
'''
    return header

def generate_jaqal_code(timesteps):
    circuit = ''
    for max in range(timesteps): 
        circuit += '// Timestep %s \n' %(max)
        for time in range(max+1):
            circuit += '// Circuit %s \n' %(time)
            circuit += 'xCD q[0] 1 1 %s %s \n' %(angles[0, time], angles[1, time])
            circuit += 'Rz q[0] %s \n' % (angles[2, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(-angles[0, time], -angles[1, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(-angles[0, time], -angles[1, time])
            circuit += 'Rz q[0] %s \n' % (angles[2, time])
            circuit += 'xCD q[0] 1 1 %s %s \n' %(angles[0, time], angles[1, time])

    return circuit


In [ ]:
#Make the jaqal files looping over the betas such that each one is a different "batch" 

circuit_list=[] # list of circuits for the subset, divided into batches
num_batch=len(beta_list)
for beta in beta_list:
    header = generate_jaqal_header(beta)
    circuits = generate_jaqal_code(timesteps)
    footer = generate_jaqal_footer()
    circuit_list.append(header + circuits + f)


In [ ]:
#This will not run on the emulator, so leaving that if statement out. 

result_list=[]
# I'm not sure why the +1 is here, removed it. - JBG
#for i in range(len(beta_list)+1):
for i in range(len(beta_list)+1):
    for jaqal_circ in circuit_list[i]: #looping over batches
        result_list[i].append(run.run_jaqal_string(jaqal_circ, overrides = override_dict))

In [ ]:
#here is where to add the readout
#an example is in the notebook Chris packaged up, let me know if you have questions. 

def generate_jaqal_footer(beta):
    footer = '''

// Characteristic-function readout.
// pi/2 rotation
loop imMeas {
    R q[1] 0 1.5707963267948966
}
xCD q[1] 1 1 reBeta imBeta
measure_all

'''

    return footer